# Notebook 4 — FiLM-Conditioned RoBERTa Baseline

This notebook tests a stronger subgroup-conditioning mechanism: **FiLM conditioning**.

Previous models provided subgroup identity either:

1. as a text prefix, or  
2. as an embedding concatenated to the final RoBERTa representation.

Both approaches allow the model to mostly ignore subgroup identity.

FiLM changes this by using subgroup identity to modulate the RoBERTa representation before prediction:

```text
h_film = gamma(subgroup) * h_text + beta(subgroup)
```

where:

- `h_text` is the RoBERTa CLS representation,
- `gamma(subgroup)` scales the text features,
- `beta(subgroup)` shifts the text features.

This notebook keeps the same data, splits, KL loss, and evaluation setup as the previous subgroup-aware baseline.


## 1. Imports and Configuration

In [2]:
import ast
import json
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42

MODEL_NAME = "roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 192
BATCH_SIZE = 16
EPOCHS = 6
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

SUBGROUP_DIM = 32
FILM_HIDDEN_DIM = 128
DROPOUT = 0.1
FILM_STRENGTH = 2.0
# Change this to "women" or "immigration"
EXPERIMENT = "women"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data")
OUTPUT_DIR = Path("women_film_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("Experiment:", EXPERIMENT)


Device: cuda
Experiment: women


In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Load Processed Data

In [4]:
women_df = pd.read_parquet(DATA_DIR / "women_processed (1).parquet")
immigration_df = pd.read_parquet(DATA_DIR / "immigration_processed.parquet")

print("Women:", women_df.shape)
print("Immigration:", immigration_df.shape)

display(women_df.head(2))
display(immigration_df.head(2))


Women: (3953, 12)
Immigration: (3799, 12)


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,6,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),train,2,"[2.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0.0,0,0.0,"{'men': 1.0, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1.0}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}"
1,11,"eat my fuck, bitch",validation,2,"[0.0, 1.0, 1.0]","[0.0, 0.5, 0.5]",1.0,1,1.5,"{'men': 1.0, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1.0}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}"


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,7,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,test,2,"[1.0, 0.0, 1.0]","[0.5, 0.0, 0.5]",1.000000,0,1.000000,"{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': 1.0, 'liberal': 1.0, 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, 'unknown': None}","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': [1.0, 0.0, 0.0], 'liberal': [0.0, 0.0, 1.0], 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, '...","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': [1.0, 0.0, 0.0], 'liberal': [0.0, 0.0, 1.0], 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, '..."
1,10,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...",train,3,"[1.0, 0.0, 2.0]","[0.3333333432674408, 0.0, 0.6666666865348816]",0.918296,2,1.333333,"{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': 1.0, 'neutral': 1.0, 'no_opinion': None, 'slightly_conservative': 1.0, 'slightly_liberal': None, 'unknown': None}","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': [0.0, 0.0, 1.0], 'neutral': [0.0, 0.0, 1.0], 'no_opinion': None, 'slightly_conservative': [1.0, 0.0, 0.0], 'slightly_libera...","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': [0.0, 0.0, 1.0], 'neutral': [0.0, 0.0, 1.0], 'no_opinion': None, 'slightly_conservative': [1.0, 0.0, 0.0], 'slightly_libera..."


## 3. Expand Comment Rows into Comment–Subgroup Examples

In [5]:
def ensure_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    raise TypeError(f"Expected dict or stringified dict, got {type(value)}")


def is_valid_distribution(dist, num_labels: int = NUM_LABELS, tolerance: float = 1e-5) -> bool:
    if dist is None:
        return False
    try:
        dist = [float(x) for x in dist]
    except Exception:
        return False
    if len(dist) != num_labels:
        return False
    if any(p < -tolerance for p in dist):
        return False
    return abs(sum(dist) - 1.0) < tolerance


def expand_subgroup_examples(
    comment_df: pd.DataFrame,
    experiment_name: str,
    allowed_subgroups: list[str] | None = None,
) -> pd.DataFrame:
    rows = []

    for _, row in comment_df.iterrows():
        subgroup_distributions = ensure_dict(row["subgroup_distributions"])
        subgroup_counts = ensure_dict(row["subgroup_counts"])

        for subgroup, target_distribution in subgroup_distributions.items():
            if allowed_subgroups is not None and subgroup not in allowed_subgroups:
                continue

            if not is_valid_distribution(target_distribution):
                continue

            target_distribution = [float(x) for x in target_distribution]

            rows.append({
                "experiment": experiment_name,
                "comment_id": row["comment_id"],
                "split": row["split"],
                "subgroup": subgroup,
                "subgroup_count": int(subgroup_counts.get(subgroup, 0)),
                "text": row["text"],
                "target_distribution": target_distribution,
                "target_majority_label": int(np.argmax(target_distribution)),
                "target_expected_label": float(np.dot(np.arange(NUM_LABELS), target_distribution)),
                "target_entropy": float(entropy(target_distribution, base=2)),
            })

    return pd.DataFrame(rows)


In [6]:
WOMEN_ALLOWED_SUBGROUPS = ["women", "men", "non_binary"]
Immigration_ALLOWED_SUBGROUPS = ["neutral","no_opinion","liberal","conservative","slightly_liberal","slightly_conservative","extremely_liberal","extremely_conservative"]
women_examples = expand_subgroup_examples(
    women_df,
    experiment_name="women",
    allowed_subgroups=WOMEN_ALLOWED_SUBGROUPS,
)

immigration_examples = expand_subgroup_examples(
    immigration_df,
    experiment_name="immigration",
    allowed_subgroups=Immigration_ALLOWED_SUBGROUPS,
)

print("Women examples:", women_examples.shape)
print("Immigration examples:", immigration_examples.shape)

display(women_examples.head())
display(immigration_examples.head())


Women examples: (7962, 10)
Immigration examples: (9090, 10)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0,0.0
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0,0.0
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0,1.0


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[1.0, 0.0, 0.0]",0,0.0,0.0
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[0.0, 0.0, 1.0]",2,2.0,0.0
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[0.0, 0.0, 1.0]",2,2.0,0.0
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[0.0, 0.0, 1.0]",2,2.0,0.0
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[1.0, 0.0, 0.0]",0,0.0,0.0


## 4. Select Experiment and Create Subgroup IDs

In [7]:
if EXPERIMENT == "women":
    model_df = women_examples.copy()
elif EXPERIMENT == "immigration":
    model_df = immigration_examples.copy()
else:
    raise ValueError("EXPERIMENT must be 'women' or 'immigration'.")

subgroup_to_id = {
    subgroup: idx
    for idx, subgroup in enumerate(sorted(model_df["subgroup"].unique()))
}

id_to_subgroup = {
    idx: subgroup
    for subgroup, idx in subgroup_to_id.items()
}

model_df["subgroup_id"] = model_df["subgroup"].map(subgroup_to_id)

print("Subgroup mapping:")
print(subgroup_to_id)

display(pd.crosstab(model_df["split"], model_df["subgroup"]))
display(model_df.head())


Subgroup mapping:
{'men': 0, 'non_binary': 1, 'women': 2}


subgroup,men,non_binary,women
split,,,
test,576,16,581
train,2723,108,2734
validation,604,16,604


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy,subgroup_id
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0,0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0,2
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0,0.0,0
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0,0.0,2
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0,1.0,0


In [8]:
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

print("Training majority-label distribution:")
display(train_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Test majority-label distribution:")
display(test_df["target_majority_label"].value_counts(normalize=True).sort_index())


Train: (5565, 11)
Val: (1224, 11)
Test: (1173, 11)
Training majority-label distribution:


target_majority_label
0    0.684277
1    0.071159
2    0.244564
Name: proportion, dtype: float64

Test majority-label distribution:


target_majority_label
0    0.669224
1    0.073316
2    0.257460
Name: proportion, dtype: float64

## 5. Dataset and Dataloader

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [10]:
class SubgroupDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "subgroup_id": torch.tensor(row["subgroup_id"], dtype=torch.long),
            "target_distribution": torch.tensor(row["target_distribution"], dtype=torch.float),
        }


In [11]:
train_dataset = SubgroupDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = SubgroupDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = SubgroupDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([16, 192]),
 'attention_mask': torch.Size([16, 192]),
 'subgroup_id': torch.Size([16]),
 'target_distribution': torch.Size([16, 3])}

## 6. FiLM-Conditioned RoBERTa Model

In [12]:
class FiLMConditionedRoBERTa(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_subgroups: int,
        subgroup_dim: int = 32,
        film_hidden_dim: int = 128,
        num_labels: int = 3,
        dropout: float = 0.1,
        film_strength: float = 2.0
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.film_strength = 2.0
        self.subgroup_embedding = nn.Embedding(
            num_embeddings=num_subgroups,
            embedding_dim=subgroup_dim,
        )

        self.film_generator = nn.Sequential(
            nn.Linear(subgroup_dim, film_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(film_hidden_dim, hidden_size * 2),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, subgroup_id):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        subgroup_embedding = self.subgroup_embedding(subgroup_id)

        film_params = self.film_generator(subgroup_embedding)
        gamma, beta = film_params.chunk(2, dim=-1)

        
        gamma = 1.0 + self.film_strength * torch.tanh(gamma)
        beta = self.film_strength * torch.tanh(beta)

        film_embedding = gamma * cls_embedding + beta

        logits = self.classifier(film_embedding)

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
            "gamma": gamma,
            "beta": beta,
        }


In [13]:
model = FiLMConditionedRoBERTa(
    model_name=MODEL_NAME,
    num_subgroups=len(subgroup_to_id),
    subgroup_dim=SUBGROUP_DIM,
    film_hidden_dim=FILM_HIDDEN_DIM,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training steps: 2088
Warmup steps: 208


## 7. Metrics

In [14]:
EPS = 1e-12


def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)


def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    values = []
    for true_dist, pred_dist in zip(y_true, y_pred):
        values.append(jensenshannon(true_dist, pred_dist, base=2) ** 2)
    return np.array(values)


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)


def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels


def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## 8. Training Helpers

In [15]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None

    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        subgroup_id = batch["subgroup_id"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                subgroup_id=subgroup_id,
            )

            loss = criterion(outputs["log_probs"], targets)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


## 9. Train Model

In [16]:
best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / f"{EXPERIMENT}_film_best_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scheduler=None,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / f"{EXPERIMENT}_film_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Epoch 1/6
Train KL: 0.7137 | Val KL: 0.6854
Train JS: 0.2859 | Val JS: 0.2418
Val Acc: 0.7075 | Val Macro F1: 0.4550
Saved new best model to women_film_outputs/women_film_best_model.pt
Epoch 2/6
Train KL: 0.6010 | Val KL: 0.6387
Train JS: 0.2312 | Val JS: 0.2292
Val Acc: 0.7214 | Val Macro F1: 0.4783
Saved new best model to women_film_outputs/women_film_best_model.pt
Epoch 3/6
Train KL: 0.5639 | Val KL: 0.6502
Train JS: 0.2168 | Val JS: 0.2306
Val Acc: 0.7214 | Val Macro F1: 0.4807
Epoch 4/6
Train KL: 0.5242 | Val KL: 0.6453
Train JS: 0.2027 | Val JS: 0.2306
Val Acc: 0.7157 | Val Macro F1: 0.4738
Epoch 5/6
Train KL: 0.4881 | Val KL: 0.6947
Train JS: 0.1901 | Val JS: 0.2292
Val Acc: 0.7198 | Val Macro F1: 0.4791
Epoch 6/6
Train KL: 0.4580 | Val KL: 0.6938
Train JS: 0.1789 | Val JS: 0.2309
Val Acc: 0.7214 | Val Macro F1: 0.4770


,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.713671,0.285943,0.794277,0.648158,0.411071,0.646824,0.083889,0.062879,0.713671,0.685377,0.241834,0.758507,0.707516,0.454955,0.520125,0.108525,0.100053,0.685377
1,2,0.600979,0.231197,0.681585,0.727942,0.474339,0.524744,0.147695,0.139435,0.600979,0.638684,0.229220,0.711814,0.721405,0.478282,0.501004,0.152334,0.139071,0.638684
2,3,0.563927,0.216758,0.644533,0.748967,0.490734,0.487175,0.172852,0.164748,0.563927,0.650214,0.230609,0.723344,0.721405,0.480665,0.497159,0.149506,0.140109,0.650214
3,4,0.524165,0.202712,0.604771,0.759030,0.499137,0.450225,0.198662,0.187303,0.524165,0.645343,0.230595,0.718473,0.715686,0.473799,0.499501,0.152847,0.138821,0.645343
4,5,0.488050,0.190076,0.568656,0.773046,0.512113,0.420671,0.216452,0.203355,0.488050,0.694658,0.229225,0.767788,0.719771,0.479068,0.491097,0.134896,0.127817,0.694658
5,6,0.458028,0.178912,0.538634,0.784007,0.518836,0.392804,0.235790,0.227210,0.458028,0.693825,0.230899,0.766955,0.721405,0.477012,0.490193,0.136682,0.128617,0.693825


Saved: women_film_outputs/women_film_training_history.csv


## 10. Test Evaluation

In [17]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scheduler=None,
)

display(test_metrics)

metrics_path = OUTPUT_DIR / f"{EXPERIMENT}_film_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:", metrics_path)


/tmp/ipykernel_30367/2391530061.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.692754864692688,
 'js_mean': 0.2431587977755616,
 'cross_entropy_mean': 0.7762179374694824,
 'accuracy': 0.7101449275362319,
 'macro_f1': 0.46523119812823793,
 'expected_label_mae': 0.5481557222265824,
 'entropy_pearson': 0.13708567908850144,
 'entropy_spearman': 0.12745105988402253,
 'loss': 0.6927549000718303}

Saved: women_film_outputs/women_film_test_metrics.json


## 11. Save Predictions

In [18]:
predictions_df = test_df.copy()

predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)

predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)

predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

display(predictions_df.head())

predictions_path = OUTPUT_DIR / f"{EXPERIMENT}_film_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

print("Saved:", predictions_path)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy,subgroup_id,true_distribution,pred_distribution,pred_majority_label,pred_expected_label,pred_entropy,kl,js,cross_entropy
0,women,63,test,men,1,Demand she pay and topple the patriarchy!,"[1.0, 0.0, 0.0]",0,0.0,0.0,0,"[1.0, 0.0, 0.0]","[0.9581079, 0.01700472, 0.024887418]",0,0.066780,0.291717,0.042795,0.021269,0.042795
1,women,63,test,women,2,Demand she pay and topple the patriarchy!,"[1.0, 0.0, 0.0]",0,0.0,0.0,2,"[1.0, 0.0, 0.0]","[0.9233583, 0.03285642, 0.043785207]",0,0.120427,0.465747,0.079738,0.039423,0.079738
2,women,193,test,men,1,I will personally rape your mother,"[0.0, 0.0, 1.0]",2,2.0,0.0,0,"[0.0, 0.0, 1.0]","[0.9473171, 0.020078663, 0.032604206]",0,0.085287,0.348200,3.423314,0.895589,3.423314
3,women,193,test,women,1,I will personally rape your mother,"[0.0, 0.0, 1.0]",2,2.0,0.0,2,"[0.0, 0.0, 1.0]","[0.8993147, 0.040262707, 0.06042255]",0,0.161108,0.568918,2.806393,0.832805,2.806393
4,women,209,test,men,1,I hope she's been sterilized,"[1.0, 0.0, 0.0]",0,0.0,0.0,0,"[1.0, 0.0, 0.0]","[0.95904034, 0.017170712, 0.023788989]",0,0.064749,0.286860,0.041822,0.020789,0.041822


Saved: women_film_outputs/women_film_test_predictions.parquet


## 12. Standard Diagnostics

In [19]:
true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())


Confusion matrix:
[[622   0 163]
 [ 48   0  38]
 [ 91   0 211]]

Classification report:
              precision    recall  f1-score   support

           0       0.82      0.79      0.80       785
           1       0.00      0.00      0.00        86
           2       0.51      0.70      0.59       302

    accuracy                           0.71      1173
   macro avg       0.44      0.50      0.47      1173
weighted avg       0.68      0.71      0.69      1173


Predicted label distribution:


0    0.648764
2    0.351236
Name: proportion, dtype: float64


True label distribution:


0    0.669224
1    0.073316
2    0.257460
Name: proportion, dtype: float64

In [20]:
print("Mean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_entropy", "mean"),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)


Mean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,785,0.345670,0.138639,0.142549,0.701107
1,86,2.610527,0.755540,0.197674,0.912026
2,302,1.048826,0.368930,0.040868,1.110883


Average predicted distribution by true majority label:
0 [0.7609587  0.05077145 0.18826985]
1 [0.6219979  0.06569536 0.3123066 ]
2 [0.44959167 0.07821441 0.4721941 ]


In [21]:
print("Performance by subgroup:")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }

    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / f"{EXPERIMENT}_film_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
print("Saved:", subgroup_metrics_path)


Performance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,non_binary,16,0.575909,0.220385,0.575909,0.812500,0.539855,0.518632,NaN,NaN
0,men,576,0.692341,0.237364,0.784935,0.701389,0.441714,0.537479,0.183225,0.156938
2,women,581,0.696383,0.249531,0.773093,0.716007,0.481650,0.559553,0.086103,0.082162


Saved: women_film_outputs/women_film_subgroup_metrics.csv


## 13. Counterfactual Subgroup Sensitivity

In [22]:
@torch.no_grad()
def predict_distribution(text: str, subgroup: str) -> np.ndarray:
    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor(
        [subgroup_to_id[subgroup]],
        dtype=torch.long,
    ).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        subgroup_id=subgroup_id,
    )

    return outputs["probs"].detach().cpu().numpy()[0]


In [23]:
subgroups = list(subgroup_to_id.keys())

counterfactual_rows = []

for _, row in test_df.drop_duplicates("comment_id").iterrows():
    text = row["text"]

    preds_by_group = {
        subgroup: predict_distribution(text, subgroup)
        for subgroup in subgroups
    }

    pairwise_js = []

    for group_a, group_b in itertools.combinations(subgroups, 2):
        js = jensenshannon(
            preds_by_group[group_a],
            preds_by_group[group_b],
            base=2,
        ) ** 2

        pairwise_js.append(js)

    counterfactual_rows.append({
        "comment_id": row["comment_id"],
        "text": text,
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_df = pd.DataFrame(counterfactual_rows)

print("Mean counterfactual JS:", cf_df["mean_pairwise_js"].mean())
print("Median counterfactual JS:", cf_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_df["max_pairwise_js"].median())

display(
    cf_df
    .sort_values("max_pairwise_js", ascending=False)
    .head(30)
)

cf_path = OUTPUT_DIR / f"{EXPERIMENT}_film_counterfactual_js.csv"
cf_df.to_csv(cf_path, index=False)
print("Saved:", cf_path)


Mean counterfactual JS: 0.005650675160325753
Median counterfactual JS: 0.006094800678465087
Mean max-pairwise JS: 0.01048077291933311
Median max-pairwise JS: 0.011063887124221718


,comment_id,text,mean_pairwise_js,max_pairwise_js
277,20465,"""Boobs and Muscles dont go together! It ruins the character!"" Me as an intellect: URL",0.016792,0.032525
83,5410,Them - DON'T YOU DARE OBJECTIFY US!! ​ Also them - Gets publicly naked as much as fucking possible.,0.016551,0.031413
506,40084,All four of these skanky narcissistic feminists need feeding to lions.,0.016738,0.031144
312,23597,men are trash and we are the bins,0.016844,0.030731
547,45258,"These 4 women are hate mongers and racists, they are actually bullying the dems. Members ,they are scared of them, these women need to be spanked! Maga",0.016080,0.030590
76,4565,Some real pieces of work in these comments. She was most likely molested by an adult because it is extremely unlikely that two 10 year olds already started puberty AND had sex with eachother. Nevertheless if that is ...,0.016245,0.030408
47,3150,Are you implying that a __true__ mgtow would pass a chance to insult women as a __species__?,0.015863,0.030196
576,49722,THESE 4 WOMEN WHO HATE THIS COUNTRY USING THE RACIST CARD EVERYDAY THESE 4 WOMEN HATE AMERICA THEY ARE SOCIALISTS RACIST DISGUSTING DISGRACE TO AMERICA DEMOCRATIC HAVE LOST IT ....TRUMP IS CORRECT OMAR TALKS HORRIBLE...,0.015679,0.029467
216,14894,Yeah dont i know it back in my day a woman didnt even have the right to say no and if she did youd give her a good wallop and these millenials woth tgeir phones and degrees buch of fucking nancies dont i know it jim ...,0.015438,0.028963
487,38838,w*men: eww ur such a loser incel virgin i bet u get no pussy and have a tiny dick kings: URL,0.015091,0.028880


Saved: women_film_outputs/women_film_counterfactual_js.csv


## 14. Pairwise Counterfactual Analysis

In [24]:
def pairwise_counterfactual_js(group_a: str, group_b: str) -> pd.DataFrame:
    rows = []

    for _, row in test_df.drop_duplicates("comment_id").iterrows():
        text = row["text"]

        pred_a = predict_distribution(text, group_a)
        pred_b = predict_distribution(text, group_b)

        js = jensenshannon(
            pred_a,
            pred_b,
            base=2,
        ) ** 2

        rows.append({
            "comment_id": row["comment_id"],
            "text": text,
            "group_a": group_a,
            "group_b": group_b,
            "js": float(js),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
        })

    return pd.DataFrame(rows)


print("Available subgroups:")
print(subgroup_to_id)


Available subgroups:
{'men': 0, 'non_binary': 1, 'women': 2}


In [25]:
if "extremely_liberal" in subgroup_to_id and "extremely_conservative" in subgroup_to_id:
    extreme_df = pairwise_counterfactual_js(
        "extremely_liberal",
        "extremely_conservative",
    )

    print("Extreme liberal vs extreme conservative mean JS:", extreme_df["js"].mean())
    print("Extreme liberal vs extreme conservative median JS:", extreme_df["js"].median())

    display(
        extreme_df
        .sort_values("js", ascending=False)
        .head(30)
    )

    extreme_path = OUTPUT_DIR / f"{EXPERIMENT}_film_EL_vs_EC_counterfactual.csv"
    extreme_df.to_csv(extreme_path, index=False)
    print("Saved:", extreme_path)

if "men" in subgroup_to_id and "women" in subgroup_to_id:
    gender_df = pairwise_counterfactual_js("men", "women")

    print("Men vs women mean JS:", gender_df["js"].mean())
    print("Men vs women median JS:", gender_df["js"].median())

    display(
        gender_df
        .sort_values("js", ascending=False)
        .head(30)
    )

    gender_path = OUTPUT_DIR / f"{EXPERIMENT}_film_men_vs_women_counterfactual.csv"
    gender_df.to_csv(gender_path, index=False)
    print("Saved:", gender_path)


Men vs women mean JS: 0.010464584853491101
Men vs women median JS: 0.011063887124221718


,comment_id,text,group_a,group_b,js,pred_men,pred_women
277,20465,"""Boobs and Muscles dont go together! It ruins the character!"" Me as an intellect: URL",men,women,0.032525,"[0.64485383, 0.08457282, 0.27057335]","[0.438518, 0.10358352, 0.45789856]"
83,5410,Them - DON'T YOU DARE OBJECTIFY US!! ​ Also them - Gets publicly naked as much as fucking possible.,men,women,0.031413,"[0.78918284, 0.05984767, 0.15096954]","[0.59881854, 0.101301774, 0.29987973]"
506,40084,All four of these skanky narcissistic feminists need feeding to lions.,men,women,0.031144,"[0.69850266, 0.072293624, 0.22920373]","[0.4975286, 0.101097845, 0.4013735]"
312,23597,men are trash and we are the bins,men,women,0.030731,"[0.6834994, 0.080368936, 0.23613167]","[0.48297203, 0.110172085, 0.40685585]"
547,45258,"These 4 women are hate mongers and racists, they are actually bullying the dems. Members ,they are scared of them, these women need to be spanked! Maga",men,women,0.030590,"[0.70958054, 0.07311742, 0.21730202]","[0.511136, 0.105552115, 0.3833119]"
76,4565,Some real pieces of work in these comments. She was most likely molested by an adult because it is extremely unlikely that two 10 year olds already started puberty AND had sex with eachother. Nevertheless if that is ...,men,women,0.030408,"[0.7279938, 0.06281932, 0.20918685]","[0.5318984, 0.09291689, 0.3751847]"
47,3150,Are you implying that a __true__ mgtow would pass a chance to insult women as a __species__?,men,women,0.030196,"[0.6936957, 0.07748261, 0.22882165]","[0.4962207, 0.104872644, 0.39890665]"
576,49722,THESE 4 WOMEN WHO HATE THIS COUNTRY USING THE RACIST CARD EVERYDAY THESE 4 WOMEN HATE AMERICA THEY ARE SOCIALISTS RACIST DISGUSTING DISGRACE TO AMERICA DEMOCRATIC HAVE LOST IT ....TRUMP IS CORRECT OMAR TALKS HORRIBLE...,men,women,0.029467,"[0.7257603, 0.07806745, 0.1961723]","[0.53235817, 0.118633255, 0.34900862]"
216,14894,Yeah dont i know it back in my day a woman didnt even have the right to say no and if she did youd give her a good wallop and these millenials woth tgeir phones and degrees buch of fucking nancies dont i know it jim ...,men,women,0.028963,"[0.6356427, 0.07934244, 0.28501487]","[0.43940622, 0.09924605, 0.46134773]"
487,38838,w*men: eww ur such a loser incel virgin i bet u get no pussy and have a tiny dick kings: URL,men,women,0.028880,"[0.6565867, 0.08505437, 0.25835896]","[0.46093923, 0.11202754, 0.42703322]"


Saved: women_film_outputs/women_film_men_vs_women_counterfactual.csv


## 15. True Subgroup Disagreement vs Model Counterfactual Disagreement

In [26]:
def valid_dist(x):
    if x is None:
        return False
    try:
        arr = np.array(x, dtype=float)
    except Exception:
        return False
    if arr.shape[0] != NUM_LABELS:
        return False
    if np.isnan(arr).any():
        return False
    if arr.sum() <= 0:
        return False
    return True


def true_pairwise_disagreement(raw_df: pd.DataFrame, group_a: str, group_b: str) -> pd.DataFrame:
    rows = []

    for _, row in raw_df.iterrows():
        subgroup_dists = row["subgroup_distributions"]

        if isinstance(subgroup_dists, str):
            subgroup_dists = ast.literal_eval(subgroup_dists)

        dist_a = subgroup_dists.get(group_a)
        dist_b = subgroup_dists.get(group_b)

        if not valid_dist(dist_a) or not valid_dist(dist_b):
            continue

        dist_a = np.array(dist_a, dtype=float)
        dist_b = np.array(dist_b, dtype=float)

        true_js = jensenshannon(dist_a, dist_b, base=2) ** 2

        rows.append({
            "comment_id": row["comment_id"],
            "text": row["text"],
            "true_js": float(true_js),
            f"{group_a}_true_dist": dist_a,
            f"{group_b}_true_dist": dist_b,
            f"{group_a}_true_label": int(np.argmax(dist_a)),
            f"{group_b}_true_label": int(np.argmax(dist_b)),
        })

    return pd.DataFrame(rows)


In [27]:
if EXPERIMENT == "immigration" and "extremely_liberal" in subgroup_to_id and "extremely_conservative" in subgroup_to_id:
    true_df = true_pairwise_disagreement(
        immigration_df,
        "extremely_liberal",
        "extremely_conservative",
    )

    comparison_df = true_df.merge(
        extreme_df[[
            "comment_id",
            "js",
            "pred_extremely_liberal",
            "pred_extremely_conservative",
        ]],
        on="comment_id",
        how="inner",
    )

    comparison_df = comparison_df.rename(columns={"js": "model_js"})

    nonzero = comparison_df[comparison_df["true_js"] > 0].copy()
    nonzero["capture_ratio"] = nonzero["model_js"] / nonzero["true_js"]

    print("N true EL/EC overlap:", len(true_df))
    print("N comparison:", len(comparison_df))
    print("Mean true JS:", comparison_df["true_js"].mean())
    print("Mean model JS:", comparison_df["model_js"].mean())
    print("Mean capture ratio, true_js > 0:", nonzero["capture_ratio"].mean())
    print("Median capture ratio, true_js > 0:", nonzero["capture_ratio"].median())

    comparison_df["label_pair"] = (
        comparison_df["extremely_liberal_true_label"].astype(str)
        + "-"
        + comparison_df["extremely_conservative_true_label"].astype(str)
    )

    display(
        comparison_df
        .groupby("label_pair")
        .agg(
            n=("comment_id", "count"),
            mean_true_js=("true_js", "mean"),
            mean_model_js=("model_js", "mean"),
        )
        .sort_values("mean_true_js", ascending=False)
    )

    display(
        comparison_df
        .sort_values("true_js", ascending=False)
        .head(30)
    )

    comparison_path = OUTPUT_DIR / f"{EXPERIMENT}_film_true_vs_model_EL_EC.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print("Saved:", comparison_path)


## 16. Interpretation Guide

Compare this FiLM model against:

1. the subgroup-token baseline,
2. the subgroup-embedding baseline,
3. the ordinal-loss run.

The most important questions are:

```text
1. Does overall KL/JS improve?
2. Does class 1 appear as a predicted class?
3. Does class 2 become less suppressed?
4. Does counterfactual subgroup sensitivity increase?
5. Does model EL↔EC JS move closer to true EL↔EC JS?
```

If FiLM increases counterfactual subgroup sensitivity even without improving overall KL, it still supports the claim that stronger identity conditioning changes model behaviour.
If FiLM also fails, that is strong evidence that the remaining disagreement requires richer contextual information rather than simple subgroup identifiers.
